In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('Oil and Gas Major Companies.csv')

In [2]:
df.info()

,Item (all numbers in Million USD unless stated otherwise),Year,Aramco,Chevron,ExxonMobil,BP,TotalEnergies,Shell
0,Net income,2018,111071.0,14824.0,20840.0,9578.0,11550.0,23352.0
1,Operating/production Cost,2018,14987.0,20544.0,36682.0,23005.0,27484.0,26970.0
2,"Avg Hydrocarbon Production, Million BPDOE",2018,13.6,2.9,3.8,2.5,2.8,3.7
3,Number of emplyees,2018,67947.0,45047.0,71000.0,73000.0,104460.0,82000.0
4,Total Equity,2018,274249.0,155642.0,198528.0,101548.0,118114.0,202534.0


## Exploratory analysis and exports

The following cells reshape the source table, compare companies and years, and export the results and written findings to this folder.

In [4]:
from pathlib import Path

output_dir = Path('.')
source_col = df.columns[0]
company_cols = [column for column in df.columns if column not in [source_col, 'Year']]
metric_names = {
    'Net income': 'Net income',
    'Operating/production Cost': 'Operating/production cost',
    'Avg Hydrocarbon Production, Million BPDOE': 'Average hydrocarbon production (million BPDOE)',
    'Number of emplyees': 'Number of employees',
    'Total Equity': 'Total equity',
    'Total Assets': 'Total assets',
    'Capital Expenditure': 'Capital expenditure'
}

tidy = df.rename(columns={source_col: 'Metric'}).copy()
tidy['Metric'] = tidy['Metric'].astype(str).str.strip().replace(metric_names)
tidy = tidy.melt(id_vars=['Year', 'Metric'], var_name='Company', value_name='Value')
tidy['Company'] = tidy['Company'].str.strip()
tidy.to_csv(output_dir / 'oil_company_tidy_data.csv', index=False)

metric_order = list(metric_names.values())
assert set(tidy['Metric']) == set(metric_order), f'Unexpected metrics: {sorted(set(tidy["Metric"]))}'
assert tidy['Value'].notna().all(), 'The analysis requires complete numeric observations.'
print(f'Analyzed {tidy["Company"].nunique()} companies, {tidy["Year"].nunique()} years, and {tidy["Metric"].nunique()} metrics.')

Analyzed 6 companies, 5 years, and 7 metrics.


In [ ]:
    "id": "f8657f95",
    "metadata": {
     "id": "f8657f95",
     "language": "python"
    },
    tidy.groupby(['Year', 'Metric'], as_index=False)['Value']
    .agg(mean='mean', median='median', minimum='min', maximum='max')
    "id": "504b3b56",
    "metadata": {
     "id": "504b3b56",
     "language": "python"
    },
trend_results = trend_results.sort_values(['Year', 'Metric'])
trend_results.to_csv(output_dir / 'oil_metric_trends.csv', index=False)
    "id": "73a0b6b5",
    "metadata": {
     "id": "73a0b6b5",
     "language": "markdown"
    },
company_wide.columns.name = None
company_wide = company_wide.sort_values(['Company', 'Year'])
    "id": "9425e4be",
    "metadata": {
     "id": "9425e4be",
     "language": "python"
    },
first_year = int(tidy['Year'].min())
last_year = int(tidy['Year'].max())
    "id": "e7b0de14",
    "metadata": {
     "id": "e7b0de14",
     "language": "python"
    },
change_rows = []
for company in company_cols:
    "id": "79172481",
    "metadata": {
     "id": "79172481",
     "language": "python"
    },
    for metric in metric_order:
        start = first_values.loc[company, metric]
        end = last_values.loc[company, metric]
        row[f'{metric} start'] = start
        row[f'{metric} end'] = end
        row[f'{metric} change'] = end - start
        row[f'{metric} change_pct'] = ((end / start) - 1) * 100 if start != 0 else np.nan
    change_rows.append(row)
change_results = pd.DataFrame(change_rows)
change_results.to_csv(output_dir / 'oil_company_period_changes.csv', index=False)

efficiency = company_wide[['Company', 'Year', 'Net income', 'Total assets', 'Total equity', 'Average hydrocarbon production (million BPDOE)']].copy()
efficiency['Return on assets (%)'] = efficiency['Net income'] / efficiency['Total assets'] * 100
efficiency['Return on equity (%)'] = efficiency['Net income'] / efficiency['Total equity'] * 100
efficiency['Net income per production unit'] = efficiency['Net income'] / efficiency['Average hydrocarbon production (million BPDOE)']
efficiency.to_csv(output_dir / 'oil_company_efficiency_metrics.csv', index=False)
print('Exported tidy data, trend statistics, period changes, and efficiency metrics.')

Exported tidy data, trend statistics, period changes, and efficiency metrics.


In [6]:
# Generate a concise, reproducible text interpretation of the results.
latest = company_wide[company_wide['Year'] == last_year].copy()
latest_income = latest.sort_values('Net income', ascending=False).iloc[0]
latest_assets = latest.sort_values('Total assets', ascending=False).iloc[0]
latest_production = latest.sort_values('Average hydrocarbon production (million BPDOE)', ascending=False).iloc[0]
latest_roe = efficiency[efficiency['Year'] == last_year].sort_values('Return on equity (%)', ascending=False).iloc[0]
income_growth = change_results.sort_values('Net income change_pct', ascending=False).iloc[0]
income_decline = change_results.sort_values('Net income change_pct').iloc[0]

report_lines = [
    'Oil and Gas Major Companies: 2018-2022 Analysis',
    '=' * 52,
    '',
    f'Dataset scope: {len(company_cols)} companies, {first_year}-{last_year}, {len(metric_order)} metrics, {len(df)} source rows.',
    'All source observations are numeric and complete; the metric labels were trimmed and standardized.',
    '',
    f'{last_year} leaders:',
    f"- Net income: {latest_income['Company']} ({latest_income['Net income']:,.1f} million USD).",
    f"- Total assets: {latest_assets['Company']} ({latest_assets['Total assets']:,.1f} million USD).",
    f"- Average hydrocarbon production: {latest_production['Company']} ({latest_production['Average hydrocarbon production (million BPDOE)']:,.2f} million BPDOE).",
    f"- Return on equity: {latest_roe['Company']} ({latest_roe['Return on equity (%)']:.2f}%).",
    '',
    f'Net income change from {first_year} to {last_year}:',
    f"- Largest percentage increase: {income_growth['Company']} ({income_growth['Net income change_pct']:.1f}%).",
    f"- Largest percentage decrease: {income_decline['Company']} ({income_decline['Net income change_pct']:.1f}%).",
    '',
    'Interpretation:',
    '- Net income and return ratios should be read alongside the major disruption in oil and gas markets during 2020-2022.',
    '- Company size and operating performance are different dimensions: assets and production rankings do not necessarily match profitability rankings.',
    '- The exported efficiency file supports further risk analysis using profitability relative to assets, equity, and production.',
    '',
    'Exported files:',
    '- oil_company_tidy_data.csv',
    '- oil_metric_trends.csv',
    '- oil_company_year_summary.csv',
    '- oil_company_period_changes.csv',
    '- oil_company_efficiency_metrics.csv',
    '- oil_analysis_report.txt'
]
report = '\n'.join(report_lines)
(output_dir / 'oil_analysis_report.txt').write_text(report, encoding='utf-8')
print(report)

Oil and Gas Major Companies: 2018-2022 Analysis

Dataset scope: 6 companies, 2018-2022, 7 metrics, 35 source rows.
All source observations are numeric and complete; the metric labels were trimmed and standardized.

2022 leaders:
- Net income: Aramco (161,068.0 million USD).
- Total assets: Aramco (664,780.0 million USD).
- Average hydrocarbon production: Aramco (13.60 million BPDOE).
- Return on equity: Aramco (36.25%).

Net income change from 2018 to 2022:
- Largest percentage increase: ExxonMobil (167.5%).
- Largest percentage decrease: BP (-114.2%).

Interpretation:
- Net income and return ratios should be read alongside the major disruption in oil and gas markets during 2020-2022.
- Company size and operating performance are different dimensions: assets and production rankings do not necessarily match profitability rankings.
- The exported efficiency file supports further risk analysis using profitability relative to assets, equity, and production.

Exported files:
- oil_company_ti